# Train YOLOv8 — VN License Plate Detector (baseline)

Chạy trên Colab với Runtime → Change runtime type → GPU.

Tham số baseline khớp với `configs/train_config.yaml` trong repo — nếu đổi tham số ở đó thì sửa lại ở đây cho khớp.

Dataset: [bomaich/vnlicenseplate](https://www.kaggle.com/datasets/bomaich/vnlicenseplate) (1 class: `plate`).

In [ ]:
!pip install -q ultralytics kagglehub

## 1. Upload Kaggle credentials

Tải `kaggle.json` từ Kaggle → Settings → API ("Create Legacy API Key"), rồi upload file khi được hỏi bên dưới.

In [ ]:
import os
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)

if not (kaggle_dir / "kaggle.json").exists():
    from google.colab import files
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    Path(uploaded_name).rename(kaggle_dir / "kaggle.json")

os.chmod(kaggle_dir / "kaggle.json", 0o600)

## 2. Tải dataset

In [ ]:
import kagglehub

dataset_path = Path(kagglehub.dataset_download("bomaich/vnlicenseplate"))
print(dataset_path)
print(list(dataset_path.iterdir()))

## 3. Tạo `data.yaml`

Dùng đường dẫn tuyệt đối trên Colab (khác với `configs/data.yaml` dùng path tương đối cho máy local).

In [ ]:
import yaml

data_yaml = {
    "path": str(dataset_path),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {0: "plate"},
}

data_yaml_path = "/content/data.yaml"
with open(data_yaml_path, "w") as f:
    yaml.dump(data_yaml, f)

print(open(data_yaml_path).read())

## 4. Train baseline (tham số khớp `configs/train_config.yaml`)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    lr0=0.01,
    patience=20,
    augment=True,
    project="runs/detect",
    name="plate_baseline",
    exist_ok=True,
)

## 5. Đánh giá trên tập validation

In [ ]:
metrics = model.val()
print(metrics)

## 6. Tải `best.pt` về máy

Sau khi tải xong, đặt file vào `models/best.pt` trong repo local, rồi ghi lại kết quả mAP/precision/recall vào `docs/pipeline.md`.

In [ ]:
from google.colab import files

best_pt = Path(model.trainer.best)
print(f"best.pt tại: {best_pt}")
files.download(str(best_pt))